# PHA 健康教练 Agent 示例

本 Notebook 展示 PHA 的**健康教练 Agent**。它会围绕健康问题与用户对话，并结合用户档案、目标和约束提供个性化建议。

## 功能

1. **对话管理**记录完整的多轮对话。
2. **记忆状态**跟踪用户目标、约束和已尝试的方法。
3. **流程控制**判断何时给出建议或结束对话。
4. **简化模式**减少记忆更新，缩短回复时间。

## 环境搭建

In [ ]:
import os
import sys

# 将工程根添加到路径
sys.path.insert(0, '..')

# --- Configuration ---
# 在此设定您的 API 密钥和提供者
API_KEY = "your-api-key-here"
# 提供方选项："基因","openai","人类"
PROVIDER = "gemini"


In [ ]:
from pha.agents import HealthCoachAgent, create_health_coach

print("已导入健康教练 Agent")

## 第 1 部分：简化模式

这种模式的配置最简单，通过减少记忆更新更快地生成回复。

In [ ]:
# 创建和配置导师
coach = HealthCoachAgent(simple_mode=True)
coach.configure(api_key=API_KEY, provider=PROVIDER) # 隐含使用全局配置

print("健康教练已在简化模式下初始化")

In [ ]:
# 单回合对话
response = coach.respond("我最近睡眠不太好。")
print("健康教练：", response)

In [ ]:
# 继续谈话
response = coach.respond("我总在凌晨 3 点左右醒来，之后就睡不着了。")
print("健康教练：", response)

In [ ]:
# 检查对话数据
print("对话统计：")
print(coach.get_stats())

In [ ]:
# 查看对话历史
print("对话记录：")
print(coach.get_conversation_history())

## 第 2 部分：完整对话模式

使用完整系统提示词，跟踪全部记忆状态。

In [ ]:
# 创建完整模式的健康教练。
coach_full = HealthCoachAgent(simple_mode=False)
coach_full.configure(api_key=API_KEY, provider=PROVIDER)

# 加载示例用户的背景信息。
coach_full.initialize_context("""
用户档案：
- 45 岁，男性。
- 从事伏案工作，每周工作超过 50 小时。
- 希望提升精力。
""")

print("已初始化完整模式并加载用户背景")

In [ ]:
# 开始面向目标的对话
response = coach_full.respond("我希望白天更有精神。")
print("健康教练：", response)

In [ ]:
# 教练会问问题
response = coach_full.respond("我通常到下午 2 点就很累，喝咖啡也不管用了。")
print("健康教练：", response)

In [ ]:
# 提供更多背景
response = coach_full.respond("我一般只睡大约 6 小时，孩子很早就会叫醒我。")
print("健康教练：", response)

In [ ]:
# 检查内存状态( 教练学到了什么)
print("对话记忆：")
print(coach_full.dump_memory())

## 第 3 部分：结合外部分析生成回复

健康教练可以利用其他来源的分析，例如编排器从专用 Agent 收集到的结果。

In [ ]:
# 创建健康教练。
coach_with_analysis = create_health_coach(
    simple_mode=True
)
coach_with_analysis.configure(api_key=API_KEY, provider=PROVIDER)

# 模拟团队中其他 Agent 提供的分析。
external_analysis = """
数据科学 Agent 的示例发现：
- 平均睡眠时长 5.8 小时，低于示例中的 7–9 小时建议区间。
- 睡眠效率 72%，低于示例中的 85% 参考值。
- 过去一个月静息心率呈上升趋势。

领域专家 Agent 的示例解读：
- 睡眠时长与用户报告的疲劳有关。
- 在考虑用药前，可先改善睡眠习惯。
- 如夜间频繁醒来，可就潜在的睡眠呼吸暂停咨询专业人员。
"""

# 让健康教练依据外部分析回答。
response = coach_with_analysis.respond_with_analysis(
    user_message="我为什么总感觉这么累？",
    analysis=external_analysis
)
print("健康教练（结合外部分析）：")
print(response)

## 第 4 部分：自定义系统提示词

In [ ]:
# 自定义健康教练的系统提示词。
custom_prompt = """
你是一位亲切的健身教练，擅长帮助工作繁忙的人。
- 提供易于实施、省时间的做法。
- 保持鼓励，同时照顾现实条件。
- 先了解时间限制，再提建议。
- 回答简短。
"""

custom_coach = HealthCoachAgent(system_prompt=custom_prompt)
custom_coach.configure(api_key=API_KEY, provider=PROVIDER)

response = custom_coach.respond("我想开始锻炼，可总是忙得抽不出时间。")
print("使用自定义提示词的健康教练：", response)

## 第 5 部分：对话流程控制

健康教练会自动判断何时给出建议或结束对话。

In [ ]:
# 演示对话结束
coach_ending = HealthCoachAgent(simple_mode=False)
coach_ending.configure(api_key=API_KEY, provider=PROVIDER)

# 说点话
coach_ending.respond("我想多喝一点水。")
coach_ending.respond("我通常一天只喝两三杯水。")
coach_ending.respond("我打算在桌上放一个水瓶试试。")

# 对话的信号结束
response = coach_ending.respond("谢谢，太有帮助了！我会试试看。")
print("健康教练：", response)

# 检查是否检测到完成
print("\n是否判断对话结束：", "FINISH" in coach_ending.memory.get("finish_desc", ""))

## 第 6 部分：交互式对话

用简单的交互循环测试健康教练。

In [ ]:
def interactive_chat(coach, max_turns=5):
    """运行一个简短的交互式对话。"""
    print("开始与健康教练对话…")
    print("（输入 quit 结束，输入 reset 重新开始，输入 memory 查看对话记忆）")
    print("-" * 50)
    
    for i in range(max_turns):
        user_input = input(f"\n你：")
        
        if user_input.lower() == 'quit':
            print("\n对话已结束，再见！")
            break
        elif user_input.lower() == 'reset':
            coach.reset_conversation()
            print("\n对话已重置，可以重新开始。")
            continue
        elif user_input.lower() == 'memory':
            print("\n对话记忆：")
            print(coach.dump_memory())
            continue
        
        response = coach.respond(user_input)
        print(f"\n健康教练： {response}")
    
    print("\n" + "-" * 50)
    print(f"对话结束，共 {coach.num_turns} 轮。")

In [ ]:
# 运行交互式聊天的解析
# coach_interactive = create_health_coach(
#     simple_mode=True
# )
# coach_interactive.configure(api_key=API_KEY, provider=PROVIDER)
# interactive_chat(coach_interactive)

## 概要

健康教练 Agent 支持自然的多轮健康对话，跟踪用户目标、约束和档案，根据对话进展给出建议或结束交流。它还可接收其他 Agent 提供的分析，以及自定义系统提示词和对话上下文。

### 后续阅读

- 查看 `05_full_pipeline.ipynb`，了解编排器如何协调各个 Agent。
- 在 `pha/prompts/health_coach_prompts.py` 中查看或调整提示词。